# 📘 [프로젝트 특강] ST1 — Streamlit 핵심 문법

> 🚀 **배포 빌드업 [1/5]**: 오늘 GitHub 저장소를 만들고 이 폴더(`apps/`)의 첫 커밋을 올립니다. ST2~ST4에서 매번 조금씩 push를 누적하고, ST5에서 최종 Cloud 배포로 마무리합니다.

## 🎯 이번 파일에서 배울 것
- `st.write`·매직(magic)으로 값을 화면에 띄우는 법 — "print의 웹 버전"
- `st.title`·`st.dataframe`·`st.metric`으로 텍스트와 표를 보여주는 법
- `st.line_chart`로 데이터를 그래프로 그리는 법
- `slider`·`selectbox`·`text_input`·`button` 위젯을 하나씩 추가하며 값이 화면에 반영되는 과정
- Streamlit의 실행 모델(rerun) — "위젯 하나 건드렸는데 왜 화면 전체가 다시 그려질까?"를 시계 트릭으로 직접 확인
- `columns`·`sidebar`·`expander`로 화면을 구성해 미니 대시보드로 조립

## 🔗 왜 지금 Streamlit인가 — 20강과의 연결
20강에서 여러분은 이미 `agent_app/app.py`를 만들었습니다. `st.text_input`으로 질문을 받고 `st.write()`로 Q&A Bot의 답을 보여주는 그 앱 — 그게 여러분의 첫 Streamlit 경험이었습니다. 다만 20강의 목표는 "Agent 로직을 완성하는 것"이었기 때문에, `app.py`가 **왜** 그렇게 동작하는지는 깊이 다루지 않았습니다.

오늘 하루는 그 반대입니다. Agent 로직은 이미 아는 것으로 치고, **그 앱의 얼굴(UI)을 제대로 이해하는 날**입니다. 왜 위젯을 조작하면 화면 전체가 다시 그려지는지 — 20강에서 "일단 되니까 넘어갔던" 질문에 이 파일에서 답합니다. (값을 rerun 사이에 기억시키는 `session_state`와, 무거운 작업을 캐싱하는 `cache_data`/`cache_resource`는 다음 파일 ST2에서 배웁니다.)

⏱️ **예상 소요 시간**: 필수 경로 약 60분 + (선택) 콘텐츠(경쟁 도구 비교·심화 `<details>`·위젯 카탈로그 정독·도전 과제) 포함 시 최대 80분. 
📋 **사전 조건**: 20강 통합 미니 프로젝트 완료(`agent_app/app.py` 실행 경험) · Python 문법 전반 · **GitHub 계정 생성 + PAT/`gh auth` 인증 1회(수업 전 사전 과제 — 당일 60분 필수 경로엔 포함되지 않습니다)**
🖥️ **실행 환경**: 커널 표시 이름(`python3` 등)과 무관하게, 실제 패키지 버전은 이 특강 전체의 `requirements.txt`(Python 3.11 기준, `streamlit==1.59.1` 등 버전 고정)를 기준으로 맞추세요. 이 파일은 `streamlit`·`pandas`·`matplotlib`·`numpy`만 쓰므로 `requirements.txt` 상단 패키지만 설치되어 있으면 됩니다.

**오늘 하루의 지도**: **ST1 Streamlit 핵심 문법(이 노트북)** → ST2 시각화·EDA 대시보드 → ST3 딥러닝 데모 앱 → ST4 에이전트 앱 → ST5 배포 & 미니 프로젝트

## 🧩 새 용어 한 줄 정의 (처음 나오는 단어부터)
아래 단어들이 파일 곳곳에 나옵니다. 지금 뜻을 완벽히 외우지 않아도 되고, 막힐 때 여기로 돌아오세요.
- **앱(app)**: `streamlit run app.py`로 띄우는, 브라우저에서 실행되는 Streamlit 스크립트 하나.
- **매직(magic)**: 문자열이나 변수를 그냥 한 줄 적기만 해도 `st.write()`를 호출한 것처럼 자동으로 화면에 표시되는 기능.
- **rerun(재실행)**: 위젯을 조작할 때마다 스크립트를 처음부터 끝까지 다시 실행하는 Streamlit의 기본 동작. (오늘의 모든 개념이 여기서 출발합니다)
- **위젯(widget)**: `st.text_input`·`st.slider`처럼 사용자 입력을 받는 화면 요소. 위젯 값은 그 rerun 시점의 값을 즉시 돌려줍니다.
- **요소(element)**: `st.title`·`st.metric`·`st.line_chart`처럼 화면에 무언가를 그리는 `st.*` 함수 각각.
- **레이아웃(layout)**: `st.sidebar`·`st.columns`·`st.expander`처럼 요소를 화면 어디에 배치할지 결정하는 구조.

> 🪟 **Windows**: 이 교안의 `python3.11 ...`은 전부 **`py -3.11 ...`**으로 바꿔 실행하세요(`수업전_환경체크.md` 참고).


## 🤔 왜 Streamlit인가 — 에이전트 개발자의 '얼굴'

**에이전트는 API만으로 팔리지 않습니다.** 17~20강에서 만든 Agent는 훌륭하게 동작하지만, 지금 그대로는 "터미널에서 `python app.py`를 칠 줄 아는 사람"만 써볼 수 있습니다. 실제로는:

- 팀장에게 "이거 한번 써보세요"라고 보여줄 **데모**가 필요하고
- 프롬프트를 바꿔가며 답변 품질을 눈으로 확인할 **검증** 도구가 필요하고
- 비개발자 동료·고객과 **공유**할 URL 하나가 필요합니다

Streamlit·Gradio·Dash 같은 프레임워크는 바로 이 "얼굴"을 만드는 도구입니다. 여러분은 20강에서 이미 얼굴 하나(`agent_app/app.py`)를 만들어봤습니다 — 오늘은 그 얼굴이 어떻게 작동하는지 이해하고, 더 잘 만드는 법을 배웁니다.

### 경쟁 도구 비교 (선택 — 시간이 빠듯하면 표만 훑고 다음 절로 이동해도 됩니다)

| 도구 | 프로그래밍 모델 | 강점 | 약점 | GitHub ⭐(참고용) |
|---|---|---|---|---|
| **Streamlit** | 스크립트 전체를 위에서 아래로 재실행 | 데이터 앱·LLM 챗봇 프로토타입, 위젯 조합 자유 | 캐싱 안 하면 성능 저하, 픽셀 단위 커스텀 어려움 | 45.1k |
| **Gradio** | 함수 시그니처 → UI 자동 매핑 | 모델 1개 데모, HF Spaces 배포 최적화 | 복잡한 멀티 페이지·레이아웃엔 약함 | 43.1k |
| **Dash** | 콜백 그래프(입력→출력 명시적 연결) | 프로덕션 BI 대시보드, 세밀한 제어 | 러닝커브가 급하고 보일러플레이트가 많음 | 24.3k |

**선택 기준**: 모델 1개를 빠르게 보여주고 싶다 → **Gradio**. 데이터와 AI 로직이 섞인 다단계 앱(오늘 우리가 만들 것) → **Streamlit**. 회사 대시보드로 프로덕션에 올린다 → **Dash**. (⭐ 수치는 참고용 스냅샷이라 분기마다 바뀝니다 — 숫자보다 "왜 이렇게 갈리는가"의 논리를 기억하세요.)

> 💡 **이전 챕터와 연결**: 15~19강에서 여러분은 LangServe·FastAPI 없이도 LLM 로직을 짤 수 있다는 걸 배웠습니다. Streamlit은 그 로직 위에 "얼굴"만 얹는 계층입니다 — 로직(15~19강)과 얼굴(오늘)을 분리해서 생각하면 어떤 프레임워크를 골라도 헤매지 않습니다.

**→ 실무 결론**: 이 특강은 Streamlit으로 진행하지만, "Streamlit이 정답"이라는 뜻은 아닙니다. 오늘 배우는 rerun·state·caching 개념은 세 프레임워크 모두에 형태만 바꿔 존재하는 보편 개념입니다 — 도구가 아니라 개념을 챙기세요.

In [ ]:
# 환경 확인 + apps/ 폴더 준비
# [왜] 오늘 만드는 모든 실습 앱은 apps/ 폴더 안에 저장합니다 — 폴더가 없으면 %%writefile이 에러를 냅니다
# [흐름] 이 구조는 20강 Step5(agent_app/ 패키지화)와 동일한 습관 — "하네스 구조는 폴더 구조에서 나온다"
import os
import streamlit as st

os.makedirs("apps", exist_ok=True)  # exist_ok=True: 이미 있어도 에러 없이 통과 — 셀 재실행 안전

print(f"✅ streamlit {st.__version__}")
print(f"✅ apps/ 폴더 준비 완료: {os.path.isdir('apps')}")

현재는 포털 내 '네이버 검색 영화 리뷰 정보' 데이터를 크롤링해야 합니다
.파이썬의 BeautifulSoup과 requests 라이브러리로 영화 평점 목록을 긁어모은 뒤, pandas를 사용해 CSV 파일로 깔끔하게 저장하는 가장 표준적인 자동화 코드를 안내해 드립니다.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

review_list = []

# [수정] 실제 영화 리뷰 데이터가 있는 네이버 쇼핑 시리즈온 주소 형식입니다.
# (예시: 영화 '기생충' 리뷰 페이지)
base_url = "https://naver.com{}"

print("🚀 네이버 영화 리뷰 크롤링을 시작합니다...")

# 1페이지부터 3페이지까지 수집 (원하는 만큼 숫자 조절 가능)
for page in range(1, 4):
    url = base_url.format(page)
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, "html.parser")
        
        # [수정] 현재 네이버 리뷰 목록을 감싸고 있는 실제 HTML 태그 경로입니다.
        reviews = soup.select(".CommentList_comment_item__3hYmX") 
        
        # 만약 페이지에 리뷰가 하나도 없다면 종료
        if not reviews:
            print(f"데이터가 더 이상 없습니다. 수집을 종료합니다.")
            break
            
        for r in reviews:
            try:
                # [수정] 평점 및 리뷰 내용 텍스트 추출 경로 맞춤
                score_element = r.select_one(".CommentItem_star_score__1N_xG")
                score = score_element.text.replace("평점", "").strip() if score_element else "10" # 평점 정보가 없으면 만점 처리
                
                comment = r.select_one(".CommentItem_content_text__2vK2x").text.strip() # 리뷰 내용
                
                review_list.append({
                    "title": "기생충", # 타겟 영화 제목
                    "sentence": comment,
                    "score": int(score)
                })
            except Exception as e:
                continue
                
        print(f"📄 {page} 페이지 수집 완료 (현재 총 {len(review_list)}건)")
        time.sleep(1.0) # 안전을 위한 1초 휴식
    else:
        print(f"❌ {page} 페이지 연결 실패 (Status Code: {response.status_code})")
        break

# 데이터프레임 변환 및 저장
df = pd.DataFrame(review_list)
output_path = "movie_data.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"🎉 크롤링 완료! 데이터가 '{output_path}' 파일로 성공적으로 저장되었습니다.")

🚀 네이버 영화 리뷰 크롤링을 시작합니다...
💥 네트워크 연결 에러 발생: HTTPSConnectionPool(host='naver.com1', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPSConnection(host='naver.com1', port=443): Failed to resolve 'naver.com1' ([Errno 11001] getaddrinfo failed)"))
❌ 수집된 데이터가 없습니다.


## 📚 1. `streamlit run` + `st.write`/매직 — "print의 웹 버전"

파이썬에서 값을 확인할 때는 `print()`를 씁니다. Streamlit에서는 그 자리를 `st.write()`가 대신합니다 — 다른 점은 터미널이 아니라 **브라우저 화면**에 찍힌다는 것뿐입니다.

```python
st.write("안녕하세요!")   # print("안녕하세요!")의 웹 버전
```

더 간단한 방법도 있습니다 — **매직(magic)**. 문자열이나 변수를 그냥 한 줄 적기만 해도 Streamlit이 자동으로 `st.write()`를 호출한 것처럼 화면에 띄워줍니다.

```python
"안녕하세요!"   # st.write("안녕하세요!")와 똑같이 동작합니다
```

💡 **비유: `st.write`는 만능 접시** — 문자열이든 숫자든 표든 그래프든, 일단 `st.write()`에 올려놓으면 Streamlit이 알아서 알맞은 모양으로 담아 보여줍니다. 모양을 세밀하게 고르고 싶을 때만 `st.title`·`st.dataframe`처럼 전용 함수를 씁니다(다음 절부터 하나씩 배웁니다).

In [ ]:
%%writefile apps/ex1_write.py
# 첫 Streamlit 앱 — st.write와 매직으로 화면에 값 띄우기
# [핵심] st.write()는 print()의 웹 버전 — 터미널 대신 브라우저 화면에 출력됩니다
import streamlit as st

st.write("안녕하세요! 이것은 저의 첫 Streamlit 앱입니다.")

# [핵심] 매직(magic) — st.write() 없이 문자열만 적어도 자동으로 화면에 표시됩니다
"매직으로도 이렇게 화면에 뜹니다 ✨"

In [ ]:
# apps/ex1_write.py 생성 확인 — "저장했다"는 로그만 믿지 않고 실제 파일을 확인합니다
from pathlib import Path

app_path = Path("apps/ex1_write.py")
if app_path.exists():
    print(f"✅ {app_path} 생성 확인 — {app_path.stat().st_size:,} bytes")
else:
    print(f"⚠️ {app_path} 가 없습니다 — 위 %%writefile 셀을 먼저 실행하세요.")

### 🚀 실행 방법 (터미널에서)

```bash
streamlit run apps/ex1_write.py
```

- 실행하면 **브라우저가 자동으로 열립니다.** 안 열리면 터미널에 뜬 `Local URL`(보통 `http://localhost:8501`)을 복사해 주소창에 붙여넣으세요.
- 앱을 **끝낼 때는 터미널에서 `Ctrl + C`** — 안 하면 다음 앱 실행 시 포트가 겹칠 수 있습니다.

**`streamlit` 명령이 `ModuleNotFoundError` 등을 내면** — 파이썬이 여러 개 설치된 PC에서 다른 파이썬의 streamlit이 실행된 것입니다. 노트북과 같은 파이썬으로 고정해서 실행하세요.

```bash
python3.11 -m streamlit run apps/ex1_write.py
```

> ⚠️ 노트북 셀 안에서 `!streamlit run ...`을 실행하면 서버가 계속 떠 있어 셀이 끝나지 않습니다. 반드시 **터미널**에서 실행하세요.

## 📚 2. 텍스트·데이터 — `st.title`/`st.markdown` → `st.dataframe`/`st.metric`

이제 `st.write` 하나 말고, 화면을 좀 더 갖춰봅니다.
- `st.title` — 큰 제목 하나 (페이지에 보통 1개만)
- `st.markdown` — 마크다운 문법(**굵게**, *기울임* 등)이 적용되는 본문 텍스트
- `st.dataframe` — 표(DataFrame)를 스크롤·정렬 가능한 형태로 출력
- `st.metric` — 숫자 하나를 카드 형태로 강조 (증감 표시도 가능)

실습: 과일 가격표 하나를 표로 띄우고, 평균가를 metric 카드로 보여줍니다.

In [ ]:
%%writefile apps/ex2_data.py
# 텍스트 + 표 + 숫자 카드 — st.title/st.markdown → st.dataframe/st.metric
# [왜] 화면에 "제목 → 설명 → 데이터"의 기본 골격을 갖추는 연습입니다
import pandas as pd
import streamlit as st

st.title("🍎 오늘의 과일 가격")
st.markdown("아래는 **표**와 **숫자 카드**로 같은 데이터를 보여준 예시입니다.")

# [흐름] 표 형태 데이터는 DataFrame으로 만들고 st.dataframe으로 그대로 띄웁니다
fruit_df = pd.DataFrame({"이름": ["사과", "바나나", "포도"], "가격": [1200, 800, 3000]})
st.dataframe(fruit_df)

# [핵심] metric은 값 하나를 카드 형태로 강조합니다 — 표보다 한눈에 들어옵니다
st.metric("오늘 평균가", f"{fruit_df['가격'].mean():.0f}원")

In [ ]:
# apps/ex2_data.py 생성 확인
from pathlib import Path

app_path = Path("apps/ex2_data.py")
if app_path.exists():
    print(f"✅ {app_path} 생성 확인 — {app_path.stat().st_size:,} bytes")
else:
    print(f"⚠️ {app_path} 가 없습니다 — 위 %%writefile 셀을 먼저 실행하세요.")

### 🚀 실행 방법
```bash
streamlit run apps/ex2_data.py
```
실행 중 오류가 나면 위 1절의 troubleshooting을 참고하세요.

## 📚 3. 차트 — `st.line_chart`/`st.bar_chart`

표 대신 그래프로 보면 추세가 한눈에 들어옵니다. `st.line_chart`는 DataFrame을 넘기기만 하면 알아서 x축(인덱스)·y축(각 열)을 그려줍니다.
```python
st.line_chart(chart_data)   # 선 그래프
st.bar_chart(chart_data)    # 막대 그래프 — 값 비교에 유리
```
실습: 랜덤 데이터로 차트를 그리고, 셀을 다시 실행할 때마다 모양이 바뀌는지 확인합니다(rerun의 예고편 — 5절에서 제대로 다룹니다).

In [ ]:
%%writefile apps/ex3_chart.py
# 랜덤 데이터 차트 — st.line_chart
# [왜] 표보다 그래프가 추세를 한눈에 보여줍니다. DataFrame만 넘기면 축은 Streamlit이 알아서 그립니다
import numpy as np
import pandas as pd
import streamlit as st

st.title("📈 랜덤 데이터 차트")

# [흐름] 20행 3열짜리 무작위 숫자표 — 매번 새로 만들어지는 데이터입니다
chart_data = pd.DataFrame(np.random.randn(20, 3), columns=["A", "B", "C"])
st.line_chart(chart_data)

# [핵심] 다시 실행하면 데이터가 바뀌는 이유는 5절(rerun 모델)에서 설명합니다
st.caption("이 앱을 다시 실행하면(재시작 또는 새로고침) 그래프 모양이 매번 바뀝니다.")

In [ ]:
# apps/ex3_chart.py 생성 확인
from pathlib import Path

app_path = Path("apps/ex3_chart.py")
if app_path.exists():
    print(f"✅ {app_path} 생성 확인 — {app_path.stat().st_size:,} bytes")
else:
    print(f"⚠️ {app_path} 가 없습니다 — 위 %%writefile 셀을 먼저 실행하세요.")

In [ ]:
# 인라인 시각화 — "rerun마다 값이 달라진다"를 이 노트북 안에서 직접 눈으로 확인
# [왜] 위 apps/ex3_chart.py는 %%writefile로 "저장"만 될 뿐 이 노트북에 그림이 뜨지 않습니다.
#      여기서는 진짜로 실행해서 렌더링까지 확인합니다 — st.line_chart(chart_data)와 똑같은 원리입니다.
# [주의] 위 5번 셀의 `import streamlit`이 matplotlib 백엔드를 비대화형 Agg로 바꿔버립니다 —
#      %matplotlib inline으로 노트북 렌더링용 백엔드를 다시 켭니다.
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import font_manager

# 한글 폰트 설정 — Mac(AppleGothic)/Win(Malgun Gothic)/Linux(NanumGothic) 중 설치된 것을 찾아 지정합니다
for _f in ["AppleGothic", "Malgun Gothic", "NanumGothic", "NanumBarunGothic"]:
    if any(_font.name == _f for _font in font_manager.fontManager.ttflist):
        plt.rcParams["font.family"] = _f
        break
plt.rcParams["axes.unicode_minus"] = False

NAVY, TEAL, CORAL = "#1F4E79", "#0E6B56", "#993C1D"
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=[NAVY, TEAL, CORAL])

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), dpi=130)
for i, ax in enumerate(axes):
    data = np.random.randn(20, 3)  # [핵심] 이 셀을 다시 실행할 때마다 여기서 새 난수가 생성됩니다
    ax.plot(data)
    ax.set_title(f"재실행 {i + 1}회차라면")
    ax.set_xlabel("스텝")
fig.suptitle("apps/ex3_chart.py의 st.line_chart(chart_data)가 이렇게 매번 다시 그려집니다", fontsize=11)
fig.tight_layout()
plt.show()

print("👉 이 셀을 Shift+Enter로 다시 실행해보세요 — 매번 그래프 모양이 바뀝니다. "
      "apps/ex3_chart.py의 line_chart도 정확히 같은 원리로, 다시 실행할 때마다 새 데이터로 다시 그려집니다.")

### 🚀 실행 방법
```bash
streamlit run apps/ex3_chart.py
```
여러 번 새로고침(F5)해서 그래프가 매번 바뀌는지 확인하세요.

## 📚 4. 위젯 — `slider` → `selectbox` → `text_input` → `button` 하나씩

위젯은 사용자 입력을 받는 화면 요소입니다. 하나씩 추가하며 값이 화면에 바로 반영되는 것을 확인합니다.
```python
level = st.slider("숙련도", 1, 10, 3)          # 1) 범위 선택
interest = st.selectbox("관심 주제", [...])     # 2) 드롭다운 선택
name = st.text_input("이름", value="수강생")   # 3) 문자열 입력
clicked = st.button("확인")                    # 4) 클릭 트리거 — 클릭 직후 rerun 1회만 True
```
모든 위젯은 **그 rerun 시점의 값을 즉시 돌려주는 함수**입니다 — 이 문장은 다음 절(rerun 모델)에서 다시 정확히 다룹니다.

In [ ]:
%%writefile apps/ex4_widgets.py
# 위젯 4종을 하나씩 추가 — slider → selectbox → text_input → button
# [왜] 위젯 값이 그 즉시 화면에 반영되는 것을 눈으로 확인하는 연습입니다
import streamlit as st

# [흐름] 위젯을 하나씩 추가하면서 st.write로 값이 반영되는지 확인합니다
st.title("🎛️ 위젯 하나씩 추가하기")

level = st.slider("숙련도 (1~10)", 1, 10, 3)                          # ① 범위 선택
interest = st.selectbox("관심 주제", ["rerun", "위젯", "레이아웃"])    # ② 드롭다운 선택
name = st.text_input("이름을 입력하세요", value="수강생")              # ③ 문자열 입력

st.write(f"{name}님, 숙련도 {level}점, 관심 주제 '{interest}'")

# [핵심] button은 클릭 직후 rerun 1회 동안만 True — 계속 누른 것처럼 보이려면 다음 파일(ST2)의 session_state가 필요합니다
if st.button("확인"):                                                  # ④ 클릭 트리거
    st.success("버튼이 클릭된 그 rerun에서만 이 메시지가 보입니다.")

In [ ]:
# apps/ex4_widgets.py 생성 확인
from pathlib import Path

app_path = Path("apps/ex4_widgets.py")
if app_path.exists():
    print(f"✅ {app_path} 생성 확인 — {app_path.stat().st_size:,} bytes")
else:
    print(f"⚠️ {app_path} 가 없습니다 — 위 %%writefile 셀을 먼저 실행하세요.")

### ⚠️ 흔한 실수 — 반복문 안에서 위젯을 여러 개 만들 때
```python
for i in range(3):
    st.button("클릭")          # ❌ 같은 key 없이 반복되면 3개 버튼이 전부 같은 ID로 충돌
```
반복문 안에서 위젯을 만들 때는 반복마다 고유한 `key`를 지정해야 합니다.
```python
for i in range(3):
    st.button("클릭", key=f"btn_{i}")   # ✅ 반복마다 고유 ID
```
`StreamlitDuplicateElementId`(예전 이름 `DuplicateWidgetID`) 에러가 이 규칙을 지키지 않을 때 발생합니다.

### 🚀 실행 방법
```bash
streamlit run apps/ex4_widgets.py
```
슬라이더·드롭다운·입력창을 하나씩 바꿔보고, 위 `st.write` 문장이 바로 바뀌는지 확인하세요.

## 📚 5. rerun 모델 — 시계 트릭으로 "왜 전체가 다시 돌지?" 확인하기

**핵심 질문: 슬라이더를 움직이면 어떤 함수가 호출될까요?**

React·Vue에 익숙하다면 `onChange` 같은 이벤트 핸들러를 찾게 됩니다. Streamlit에는 **기본적으로 그런 게 없습니다.**

Streamlit의 실행 모델은 훨씬 단순합니다.

> **위젯을 조작한다 → 스크립트 파일 전체가 위에서 아래로 처음부터 다시 실행된다("rerun") → 그 시점 위젯 값이 변수에 담겨 있다**

`name = st.text_input(...)` 이 줄은 "이벤트가 오면 name을 갱신하라"는 예약이 아니라, **이 rerun에서 text_input이 갖고 있던 값을 그냥 즉시 돌려주는 함수 호출**입니다. 다음 rerun에서는 이 줄이 다시 실행되고, 그때의 최신 값을 돌려줍니다.

**시계 트릭 — 오늘 하루 종일 쓸 디버깅 도구**: 아래 앱의 캡션에 `time.strftime("%H:%M:%S")`를 넣어두면, 위젯을 하나만 건드려도 캡션의 시각이 바뀌는 걸 볼 수 있습니다. 시각이 바뀐다 = 스크립트가 처음부터 다시 돌았다는 증거입니다. "왜 내 코드가 다시 실행되지?" 같은 의문이 들 때마다 이 트릭을 떠올리세요. (단, 초 단위라 같은 초 안에 두 번 조작하면 시각이 그대로일 수 있습니다 — 그럴 땐 1초 정도 텀을 두고 다시 눌러보세요.)

**★ 오개념 교정 — "슬라이더를 움직이면 그 위젯만 다시 그려진다"는 틀립니다**: 화면에 보이는 변화는 부분적이지만, 내부적으로는 **스크립트 전체**가 재실행됩니다. 무거운 모델 로딩이나 대용량 계산이 캐싱 없이 최상단에 있으면, 슬라이더를 만질 때마다 그 무거운 코드가 매번 다시 실행됩니다 — 이걸 다음 파일(ST2)에서 캐싱으로 해결하는 법을 배웁니다.

*(참고: 오늘 배우는 "위젯 조작 → 전체 rerun"은 **기본** 동작입니다. 콜백(`on_click`/`on_change`)이나 `st.fragment`를 쓰면 예외가 생기는데, on_click 콜백은 ST2 심화에서 다룹니다. 오늘은 가장 단순한 기본 동작만 다룹니다.)*

💡 **비유: 엑셀은 똑똑하게 재계산하지만, Streamlit은 그렇지 않다**

> 엑셀은 셀 하나를 바꾸면 의존성 그래프를 따라가 **바뀐 셀에 딸린 것만** 똑똑하게 다시 계산합니다. 그런데 Streamlit엔 **기본적으로** 그런 스마트함이 없습니다 — 위젯 하나만 건드려도 스크립트를 처음부터 끝까지 **전부** 다시 실행합니다(콜백과 `st.fragment`라는 예외는 있습니다 — 다음 파일 ST2에서 다룹니다). 화면엔 바뀐 결과만 보이니 부분 업데이트처럼 느껴지지만, 내부적으로는 매번 전체가 다시 돌아갑니다. 이 스마트함의 부재가 바로 다음 파일(ST2)에서 캐싱을 배우는 이유입니다 — 무거운 연산을 캐싱해두지 않으면 그 연산이 위젯 하나 건드릴 때마다 처음부터 다시 실행되니까요.

<details>
<summary>🔬 <b>심화 [개발자 트랙]</b>: 전체 재실행인데 화면은 왜 부분만 바뀌나</summary>

- Streamlit은 세션마다 <b>ScriptRunner</b> 스레드를 두고, 상호작용이 생기면 스크립트를 위에서 아래로 <b>전체 재실행</b>합니다.
- 위젯은 (타입 + 파라미터 + 페이지/폼 컨텍스트)의 <b>해시</b>로 식별됩니다. 이 해시엔 "코드상 몇 번째 호출인지(위치)"는 <b>포함되지 않습니다</b> — 그래서 <code>for</code> 루프에서 파라미터가 같은 위젯을 반복 생성하면 전부 같은 ID로 계산돼 충돌합니다(아래 '흔한 실수' 참고). <code>key=</code>를 주면 그 값이 해시에 섞여 위치 대신 구분자 역할을 합니다.
- 재계산한 element tree를 직전 렌더와 비교(reconcile)해 <b>바뀐 부분만 delta 메시지</b>로 브라우저에 보냅니다. "전체 재실행"과 "부분 화면 갱신"이 모순 없이 공존하는 이유입니다.
</details>

In [ ]:
%%writefile apps/ex5_clock.py
# 시계 트릭 — 위젯을 하나만 건드려도 스크립트 전체가 다시 실행되는 것을 확인
# [왜] 시각이 바뀐다 = rerun이 일어났다는 증거. 오늘 하루 종일 쓰는 디버깅 도구입니다
import time
import streamlit as st

st.title("🕐 시계 트릭")
# [핵심] 위젯을 하나만 만져도 이 캡션의 시각이 바뀝니다
st.caption(f"마지막 실행 시각: {time.strftime('%H:%M:%S')}")

level = st.slider("아무 값이나 움직여보세요", 1, 10, 3)
st.write(f"현재 값: {level}")

In [ ]:
# apps/ex5_clock.py 생성 확인
from pathlib import Path

app_path = Path("apps/ex5_clock.py")
if app_path.exists():
    print(f"✅ {app_path} 생성 확인 — {app_path.stat().st_size:,} bytes")
else:
    print(f"⚠️ {app_path} 가 없습니다 — 위 %%writefile 셀을 먼저 실행하세요.")

### 🚀 실행 방법
```bash
streamlit run apps/ex5_clock.py
```
슬라이더를 움직일 때마다 캡션의 시각이 바뀌는지 확인하세요 — 바뀐다면 스크립트 전체가 다시 실행된 것입니다. (3절의 랜덤 차트가 매번 모양이 바뀌었던 것도 같은 이유입니다.)

## 📚 6. 레이아웃 — `columns`·`sidebar`·`expander`로 미니 대시보드 조립

지금까지 배운 걸 모두 모아 하나의 대시보드로 조립합니다. 레이아웃 요소 3가지만 더 배우면 됩니다.
- `st.columns(n)` — 화면을 가로로 n등분
- `st.sidebar` — 본문과 분리된 설정 전용 영역
- `st.expander` — 자주 안 쓰는 내용을 접어 화면을 간결하게
- (덤) `st.tabs` — 같은 화면에서 탭으로 콘텐츠 전환
- (보조 위젯) `st.checkbox`·`st.radio` — 4절에서 배운 위젯 감(感)을 레이아웃 안에서 한 번 더 씁니다(자세한 설명은 7절 위젯 카탈로그 참고)

아래 `apps/m1_hello.py`가 이번 절의 완성본입니다 — 오늘 배운 write/매직·표·차트·위젯·시계 트릭·레이아웃이 전부 들어 있습니다.

> **📝 `%%writefile`은 실행이 아니라 파일 저장입니다.** 이 셀을 실행하면 코드가 돌아가는 게 아니라 `apps/m1_hello.py` 파일이 만들어집니다. 초록색 `Writing apps/m1_hello.py` 메시지가 뜨면 성공 — 앱은 아래 안내대로 **터미널**에서 따로 실행합니다.

In [ ]:
%%writefile apps/m1_hello.py
# 첫 Streamlit 앱 — 실행 모델(rerun)을 눈으로 확인하는 용도
# [권장] st.set_page_config는 import 직후 첫 st 명령으로 두는 것이 관례입니다(브라우저 탭 제목·레이아웃을 먼저 확정)
#        참고: 이 특강의 Streamlit 1.59.1에선 순서가 어긋나거나 여러 번 호출해도 에러 없이 additive하게 적용됩니다
import time
import numpy as np
import pandas as pd
import streamlit as st

st.set_page_config(page_title="첫 Streamlit 앱", page_icon="🎈", layout="centered")

# [핵심] 제목은 페이지에 보통 하나만 둡니다
st.title("🎈 나의 첫 Streamlit 앱")
# [핵심] 시계 트릭 — 위젯을 하나만 건드려도 이 시각이 바뀝니다. 바뀐다 = 스크립트 전체가 다시 실행됐다는 증거
st.caption(f"마지막 실행 시각: {time.strftime('%H:%M:%S')}")

# [widget] 위젯 3종 — text_input(문자열)·slider(범위)·selectbox(목록) 순서로 값을 받습니다
name = st.text_input("이름을 입력하세요", value="수강생")
level = st.slider("Streamlit 숙련도 (1~10)", 1, 10, 3)
interest = st.selectbox("오늘 가장 궁금한 주제", ["실행 모델(rerun)", "session_state", "캐싱", "배포"])

# [핵심] 위젯 값은 그 rerun 시점의 값을 즉시 돌려주는 변수일 뿐입니다(5절 참고)
st.write(f"안녕하세요, **{name}**님! 오늘 숙련도 {level}점으로 시작해서 '{interest}'를 배웁니다.")

# [흐름] 매 rerun마다 새 난수가 생성됩니다 — 슬라이더를 움직일 때마다 그래프 모양이 바뀌는 것으로도 rerun을 확인할 수 있습니다
chart_data = pd.DataFrame(np.random.randn(20, 3), columns=["A", "B", "C"])

# [layout] columns — 화면을 가로로 n등분해 나란히 배치할 때 씁니다
col1, col2 = st.columns(2)
col1.metric("현재 숙련도", level, delta=level - 3)
col2.metric("목표 숙련도", 10, delta=10 - level)

# [layout] tabs — 같은 데이터를 화면 전환으로 보여줄 때 씁니다
tab_chart, tab_table = st.tabs(["📈 차트", "📋 통계 요약"])
with tab_chart:
    st.line_chart(chart_data)
with tab_table:
    st.dataframe(chart_data.describe())

# [layout] expander — 자주 안 쓰는 옵션을 접어둘 때 씁니다. checkbox — 켜기/끄기 토글
with st.expander("⚙️ 고급 설정 (평소엔 접어두고, 필요할 때만 펼침)"):
    show_raw = st.checkbox("원본 데이터 20행 보기")
    if show_raw:
        st.dataframe(chart_data)

# [흐름] 사이드바는 본문과 분리된 별도 영역 — 설정용 위젯을 모아두는 관용적 위치
st.sidebar.header("⚙️ 오늘의 설정")
st.sidebar.write(f"선택한 이름: {name}")
st.sidebar.write(f"관심 주제: {interest}")
# [widget] radio — 선택지가 2~4개로 적을 때 selectbox 대신 씁니다(항상 다 보여서 클릭 한 번에 선택)
study_mode = st.sidebar.radio("오늘 학습 방식", ["실습 위주", "이론 위주"])
st.sidebar.caption(f"선택: {study_mode}")

In [ ]:
# apps/m1_hello.py 생성 확인 — "저장했다"는 로그만 믿지 않고 실제 파일을 확인합니다
from pathlib import Path

app_path = Path("apps/m1_hello.py")
if app_path.exists():
    print(f"✅ {app_path} 생성 확인 — {app_path.stat().st_size:,} bytes")
else:
    print(f"⚠️ {app_path} 가 없습니다 — 위 %%writefile 셀을 먼저 실행하세요.")

### 🚀 실행 방법
```bash
streamlit run apps/m1_hello.py
```
탭(📈 차트 / 📋 통계 요약)을 전환해보고, 사이드바 라디오·expander도 눌러보세요. 위젯을 하나 바꿀 때마다 캡션의 시각이 바뀌는지도 다시 확인해보세요(5절 시계 트릭).

### ⚠️ 흔한 실수·오해

**`set_page_config`는 반드시 첫 명령" — 옛날 규칙입니다 (오해)**
```python
st.title("제목")
st.set_page_config(...)   # 예전 Streamlit은 여기서 에러를 냈습니다
```
많은 블로그·예제가 "`set_page_config`가 첫 `st.*` 명령이 아니면 에러(`StreamlitSetPageConfigMustBeFirstCommandError`)"라고 설명합니다. 하지만 **우리가 쓰는 Streamlit 1.59.1에선 순서가 어긋나거나 여러 번 호출해도 에러 없이 적용(additive)됩니다** — 그 에러 클래스 자체가 지금은 존재하지 않습니다. 버전마다 동작이 다르니 "**내 버전에서 직접 확인**"하는 습관이 중요합니다(오늘의 실측주의!). 다만 브라우저 탭 제목·레이아웃을 먼저 확정하려면 여전히 맨 위에 두는 것이 관례입니다.

### ✅ 체크포인트 — 스스로 답해보세요
1. 슬라이더를 하나만 움직였는데 화면의 시계 캡션도 바뀝니다. 왜일까요?
2. 옛 자료들은 `st.set_page_config()`를 `st.title()` 다음 줄에 쓰면 에러가 난다고 합니다. 우리가 쓰는 1.59.1에서도 그럴까요?
3. `for` 반복문 안에서 버튼을 3개 만들려면 무엇을 추가해야 할까요?

<details><summary>정답 보기</summary>

1. 위젯 조작은 특정 함수를 호출하는 게 아니라 **스크립트 전체를 처음부터 다시 실행**시킵니다. 캡션 코드도 다시 실행되니 시각이 갱신됩니다.
2. **아니요.** 1.59.1에선 순서가 어긋나거나 여러 번 호출해도 에러 없이 additive하게 적용됩니다(그 에러 클래스 자체가 지금은 없습니다). 예전엔 에러였지만 지금은 아닙니다 — "내 버전에서 직접 확인"이 정답을 가릅니다. (가독성상 맨 위 두는 것은 여전히 권장.)
3. 각 버튼에 `key=f"btn_{i}"`처럼 반복마다 다른 고유 key를 지정해야 `StreamlitDuplicateElementId`(예전 이름 `DuplicateWidgetID`) 충돌을 피할 수 있습니다.

</details>

---
## 💡 추가로 시도해볼 것 (AI Pair와 함께) — Streamlit 핵심 문법

오늘 배운 rerun·위젯·레이아웃을 **부담 없이** 코드로 짜본 뒤, AI에게 리뷰를 받는 순서로 진행합니다. 이 파일은 실습형이라 Solo는 딱 1개만 필수로 풀고, 나머지 조합·확장은 아래 '📌 오늘의 도전 과제(선택)'에서 원하는 만큼 골라 집에서 시도하세요.

| 단계 | 내가 하는 것 | AI가 하는 것 |
|---|---|---|
| 시도해보기(필수) | 아래 카운터 문제를 직접 작성 | (아직 사용 X) |
| AI 리뷰받기(선택) | 내가 짠 코드를 그대로 제출 | 코드 리뷰 + 개선안 제시 |

### 시도해보기 — 지금 바로 하나만 풀어보세요 (필수)

💡 *AI에게 물어보면 30초 만에 답 코드가 나옵니다. 하지만 지금 막히는 지점 — "왜 카운터가 늘어나지 않는지" — 이야말로 오늘 배운 rerun 모델이 손에 익었는지 확인하는 지점입니다. 막히면 위 5절의 코드 셀을 다시 열어 같은 패턴을 찾아보고, 막히면 바로 AI에게 물어봐도 됩니다 — 시도 자체가 배움입니다.*

**실습 코드 셀 진행법** (아래 셀은 정답이 아니라 "뼈대 + 힌트 주석 + `pass`"입니다):
1. 셀 상단의 힌트 주석과 "몇 절 참고" 표시를 먼저 읽으세요.
2. `pass` 자리를 자신의 코드로 교체하세요.
3. 완성했다면 셀을 실행해 파일로 저장한 뒤 `streamlit run apps/ex_*.py`로 직접 동작을 확인하세요.

### Step A — 실패하는 카운터 (5절 rerun 모델을 손으로 확인)
`st.button`을 누르면 `count`가 1씩 늘어나야 할 것 같은 카운터를 만들되, **아직 `session_state`는 쓰지 마세요**(다음 파일 ST2에서 배웁니다). 버튼을 몇 번 눌러도 화면엔 계속 같은 값만 보이는 이유를 5절의 rerun 모델로 설명할 수 있어야 합니다.

In [ ]:
%%writefile apps/ex_failcounter.py
# 🔨 Step A: 실패하는 카운터 — 완성 후 `streamlit run apps/ex_failcounter.py` (5절 rerun 모델 참고)
# [패턴] session_state 없이 카운터를 만들어, 왜 값이 유지되지 않는지 rerun 모델로 직접 확인하는 문제입니다.
import time
import streamlit as st

st.title("Step A — 실패하는 카운터")

# [핵심] 시계 트릭 — 버튼을 누를 때마다 이 시각이 바뀌는지 확인하세요(=rerun이 일어났다는 증거)
st.caption(f"마지막 실행 시각: {time.strftime('%H:%M:%S')}")

# ① count를 0으로 두고, 버튼을 누르면 1 증가시켜 st.write로 보여주세요 (session_state 사용 금지)
# 힌트: count = 0
#      if st.button("증가"):
#          count += 1
#      st.write(count)
pass  # ← 여기에 실패하는 카운터 코드를 작성

# ② 버튼을 5번 눌러보고, count가 왜 1을 넘지 못하는지 위 5절 내용으로 설명해보세요(코드 주석으로 남겨도 좋습니다)

---
### 📌 오늘의 도전 과제 (선택) — 집에서 바이브코딩으로 시도해보세요

위 Solo에서 rerun의 함정을 손으로 확인했다면 충분합니다. 아래는 오늘 배운 위젯·레이아웃을 더 조합해보고 싶을 때, 원하는 만큼 골라 시도하는 확장 문제입니다 — 전부 풀 필요는 없습니다.

1. **위젯 3종 조합** — `st.text_input`·`st.slider`·`st.selectbox` 값을 한 문장으로 `st.write` 출력 (4절 참고)
2. **사이드바 + columns 대시보드** — 사이드바에 설정 위젯 1개, 본문은 `st.columns`로 2단, 각 칸에 `st.metric` (6절 참고)
3. **라디오로 차트 종류 전환** — 위 `apps/m1_hello.py`를 직접 수정해, 사이드바 radio로 line/bar/area 차트를 전환해보세요.
   ```python
   chart_type = st.sidebar.radio("차트 종류", ["line", "bar", "area"])
   if chart_type == "line":
       st.line_chart(chart_data)
   elif chart_type == "bar":
       st.bar_chart(chart_data)
   else:
       st.area_chart(chart_data)
   ```

4. **막힌 부분, AI에게 힌트만 요청하기** — 위 도전 과제 중 하나가 막히면, 정답 대신 힌트 1단계만 받는 프롬프트를 시도해보세요.
   ```text
   아래 Streamlit 코드에서 [막힌 부분: 예) 버튼을 눌러도 카운터가 늘어난 것처럼 안 보임]을 해결하려는데 잘 안 됩니다.
   정답을 바로 알려주지 말고, 힌트 1단계만 먼저 알려주세요.

   목표: [무엇을 하려는지 한 줄로]
   현재 코드:
   (여기에 막힌 코드를 붙여 넣으세요)

   증상:
   (버튼을 눌렀을 때 기대한 동작과 실제 동작의 차이를 적으세요)
   ```

아래 코드 셀은 1·2번의 뼈대입니다(3번은 위 `m1_hello.py`를 직접 고치는 문제, 4번은 프롬프트 템플릿이라 별도 파일이 없습니다) — 원하는 것만 골라 채우세요.

In [ ]:
%%writefile apps/ex_widgets3.py
# 🔨 도전 1: 위젯 3종 조합 — 완성 후 `streamlit run apps/ex_widgets3.py` (4절 참고)
# [패턴] text_input · slider · selectbox 값을 한 문장으로 조합해 st.write로 출력하는 문제입니다.
import streamlit as st

st.title("도전 1 — 위젯 3종 조합")

# ① 이름을 입력받는 text_input을 만드세요
# 힌트: name = st.text_input("이름을 입력하세요", value="수강생")
name = None  # ← 여기에 text_input 코드를 작성

# ② 1~10 사이 숙련도를 받는 slider를 만드세요
# 힌트: level = st.slider("숙련도", 1, 10, 3)
level = None  # ← 여기에 slider 코드를 작성

# ③ 관심 주제를 고르는 selectbox를 만드세요
# 힌트: interest = st.selectbox("관심 주제", ["rerun", "위젯", "레이아웃"])
interest = None  # ← 여기에 selectbox 코드를 작성

# ④ 위 세 값을 한 문장으로 조합해 st.write로 출력하세요
# 힌트: st.write(f"{name}님은 숙련도 {level}점, '{interest}'에 관심이 있습니다.")
pass  # ← 여기에 출력 코드를 작성

In [ ]:
%%writefile apps/ex_dashboard.py
# 🔨 도전 2: 사이드바 + columns 대시보드 뼈대 — 완성 후 `streamlit run apps/ex_dashboard.py`
# [패턴] 사이드바(설정) + columns(본문 2단 배치) + metric 조합 문제입니다.
import streamlit as st

st.title("도전 2 — 대시보드 뼈대")

# ① 사이드바에 슬라이더나 selectbox 등 설정용 위젯을 1개 이상 만드세요
# 힌트: target = st.sidebar.slider("목표 값", 0, 100, 50)
target = None  # ← 여기에 사이드바 위젯 코드를 작성

# ② st.columns(2)로 본문을 2단으로 나누세요
# 힌트: col1, col2 = st.columns(2)
col1, col2 = None, None  # ← 여기에 columns 코드를 작성

# ③ 각 칸에 st.metric을 하나씩 표시하세요 (하나는 ①의 target 값을 활용)
# 힌트: col1.metric("현재 값", 42)
#      col2.metric("목표 값", target)
pass  # ← 여기에 metric 표시 코드를 작성

---
### AI 리뷰받기 — 내가 만든 코드를 AI에게 리뷰받기 (선택)

Solo나 도전 과제에서 작성한 코드를 그대로 복사해서, 아래 프롬프트와 함께 AI(ChatGPT / Claude / Gemini)에게 보내 보세요. **이것이 이후 여러분이 showcase를 바이브코딩으로 고도화할 때 쓰는 바로 그 프롬프트 패턴입니다.**

📝 **프롬프트 카드 — Streamlit 코드 리뷰**

```text
당신은 Streamlit을 가르치는 멘토입니다.
아래는 제가 작성한 Streamlit 코드입니다. 다음 3가지를 확인해 주세요.

1. 위젯 값을 rerun 시점의 값으로 올바르게 다루고 있는지 (이벤트 핸들러처럼 착각한 부분이 없는지)
2. 반복문 안에서 위젯을 만든다면 고유한 key를 지정했는지 (StreamlitDuplicateElementId 방지)
3. columns·sidebar·expander 같은 레이아웃을 화면 목적에 맞게 썼는지

===== 내 코드 =====
(여기에 작성한 코드를 붙여 넣으세요)
==================
```

(막힌 부분만 콕 집어 힌트를 받는 프롬프트는 위 "오늘의 도전 과제(선택)" 4번에 있습니다.)

🎯 **마무리 체크**
- [ ] Step A의 "실패하는 카운터"가 몇 번을 눌러도 값이 유지되지 않는 걸 직접 눈으로 봤다
- [ ] 왜 그런지 rerun 모델로 설명할 수 있다(다음 파일 ST2에서 `session_state`로 같은 유형의 문제를 고치는 법을 배웁니다)
- [ ] 도전 과제 중 최소 하나를 시도했다(선택)
- [ ] AI 리뷰를 받았다면, 제안을 **직접 실행해서** 검증했다

## 📚 7. 위젯 카탈로그(참조표) + 배포 준비

지금까지 위 데모에서 여러 위젯을 이미 써봤습니다. 아래는 오늘 쓴 위젯·레이아웃 전체를 한 장으로 정리한 표입니다 — 이름이 기억 안 나면 여기로 돌아오세요.

| 카테고리 | 위젯 / 함수 | 한 줄 설명 |
|---|---|---|
| 텍스트 입력 | `st.text_input` | 한 줄 문자열 입력 — 4절 데모의 이름 입력 |
| 범위 선택 | `st.slider` | 최소~최대 사이 값을 드래그로 선택 — 4절 데모의 숙련도 |
| 단일 선택(드롭다운) | `st.selectbox` | 목록 중 하나를 드롭다운으로 선택 |
| 단일 선택(항상 노출) | `st.radio` | 목록 중 하나를 라디오 버튼으로 선택 — 옵션이 2~4개로 적을 때 |
| 켜기/끄기 | `st.checkbox` | 토글 — 6절 데모의 "원본 데이터 보기" |
| 실행 트리거 | `st.button` | 클릭 직후 rerun 1회 동안만 `True`를 돌려줌 |
| 파일 업로드 | `st.file_uploader` | 파일 업로드 위젯 — 본격 사용은 ST3(딥러닝 데모)의 이미지 업로드에서 |
| 화면 분할 | `st.columns` | 본문을 가로로 n등분 — 6절 데모의 metric 2단 |
| 별도 영역 | `st.sidebar` | 본문과 분리된 설정 전용 영역 |
| 화면 전환 | `st.tabs` | 같은 화면 안에서 탭으로 콘텐츠 전환 — 6절 데모의 차트/통계 탭 |
| 접기/펼치기 | `st.expander` | 자주 안 쓰는 내용을 접어 화면을 간결하게 |
| 숫자 강조 | `st.metric` | 값 + 증감(delta)을 카드형으로 표시 — 2절·6절 |
| 표 출력 | `st.dataframe` | 스크롤·정렬 가능한 표 — 2절 |
| 차트(선/막대/영역) | `st.line_chart` / `st.bar_chart` / `st.area_chart` | 연속값 추세를 그림 — 3절 |
| 진행 표시 | `st.progress` | 0.0~1.0 사이 값으로 진행률 바 표시 — 오래 걸리는 작업(파일 처리·모델 추론)의 진행 상황을 알려줄 때 |

**→ 실무 결론**: 위젯 이름을 다 외울 필요는 없습니다. "지금 화면에 뭘 놓고 싶은가"부터 정하고, 이 표에서 카테고리로 찾으세요 — 실제로도 대부분의 개발자가 이렇게 씁니다.

---
## 🚀 배포 빌드업 [1/5] — GitHub 저장소 만들고 첫 앱 올리기

ST5에서 오늘 만든 앱들을 Streamlit Cloud에 배포하려면, 먼저 코드가 **GitHub 저장소**에 있어야 합니다. 오늘은 저장소를 만들고 지금까지 만든 `apps/` 폴더를 첫 커밋으로 올립니다 — ST2~ST4에서는 각 파일 끝에 변경분만 한 줄로 추가 push하고, ST5에서 최종 배포로 마무리합니다.

### git 기본 명령 (터미널/PowerShell 공통)
```bash
# 1. 저장소 초기화 (최초 1회)
git init
git add apps/

# 이후 확인용 — 커밋 전 항상 이걸로 뭐가 올라가는지 확인하는 습관을 들이세요
git status

git commit -m "ST1: 첫 Streamlit 앱과 기본 문법 실습"

# 2. GitHub 웹사이트에서 빈 저장소를 하나 만든 뒤(New repository), 연결
git remote add origin https://github.com/<내계정>/<저장소명>.git
git branch -M main
git push -u origin main
```

**인증(수업 전 사전 과제)**: 첫 `git push` 시 GitHub은 비밀번호 대신 **Personal Access Token**을 요구합니다 — 계정 생성과 토큰 발급(또는 `gh auth login` 인증)은 오늘 60분 필수 경로에 포함되지 않으니 **수업 전에 미리 끝내두세요**(GitHub → Settings → Developer settings → Personal access tokens에서 발급, 비밀번호 입력란에 토큰을 붙여넣으면 됩니다. 또는 `gh auth login`으로 GitHub CLI 인증을 한 번 끝내면 이후 push가 더 매끄럽습니다). 오늘 수업 중에는 이미 인증된 상태로 git push만 실행하면 됩니다.

⚠️ **보안 — `.env`·`secrets.toml`은 절대 커밋 금지**: `.gitignore`에 두 파일을 미리 추가해두고, 커밋 전 `git status`로 커밋 대상에 들어있지 않은지 매번 확인하는 습관을 들이세요. 한 번 push된 비밀 키는 저장소를 비공개로 바꿔도 히스토리에 남습니다.

**폴백(웹 업로드)**: git이 아직 부담스럽다면, GitHub 웹사이트에서 저장소를 만든 뒤 **"Add file → Upload files"**로 `apps/` 폴더의 `.py` 파일들을 하나씩 드래그해 올리는 방법도 있습니다(이 방법도 `.env` 파일은 절대 올리지 마세요). 오늘은 어느 방법이든 **저장소에 첫 커밋을 남기는 것**이 목표입니다.

📖 자세한 단계는 `../deploy/배포_튜토리얼.md` 참고

## 📌 정리 — 다음 파일 예고

오늘 배운 것: `st.write`/매직으로 값 띄우기, `st.title`·`st.dataframe`·`st.metric`으로 텍스트·데이터 보여주기, `st.line_chart`로 차트 그리기, `slider`·`selectbox`·`text_input`·`button` 위젯 하나씩 추가하기, 그리고 Streamlit의 실행 모델(rerun)을 시계 트릭으로 확인한 뒤 `columns`·`sidebar`·`expander`로 미니 대시보드까지 조립했습니다.

**다음 파일**: **ST2(시각화·EDA 대시보드)**에서는 오늘 궁금했던 두 가지 — "값을 rerun 사이에 어떻게 기억시키지?"(`session_state`)와 "무거운 작업을 어떻게 캐싱하지?"(`cache_data`/`cache_resource`) — 를 펭귄 데이터 대시보드 맥락에서 바로 배웁니다. 오늘 배운 rerun 모델이 그 출발점입니다.